# Experiment 5: Subword Level Tokenization, POS Tagging and Word2Vec

## Experiment 5.1a: BPE tokenizer using a pretrained model (GPT-2)
Read text from file, tokenize it into subword units using the pretrained GPT-2 byte-level BPE tokenizer, and display the tokens with their token IDs. Also test the tokenizer on new words.

In [1]:
!pip install transformers -q

from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

with open("input_sub_word_data.txt", "r", encoding="utf-8") as f:
    text = f.read()

tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("Total tokens:", len(tokens))
print("\nFirst 40 tokens and IDs:")
for t, i in list(zip(tokens, ids))[:40]:
    print(f"{t:<20}{i}")

test_words = ["naturalization", "internationalization", "computationally", "unbelievable",
              "preprocessing", "classification", "communication", "renewable",
              "hydrokinetic", "multilingual", "understanding", "modernization"]

print("\nTest words:")
for w in test_words:
    toks = tokenizer.tokenize(w)
    print(f"{w:<22}{toks}  {tokenizer.convert_tokens_to_ids(toks)}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1348 > 1024). Running this sequence through the model will result in indexing errors


Total tokens: 1348

First 40 tokens and IDs:
Natural             35364
Ġlanguage           3303
Ġprocessing         7587
Ġis                 318
Ġa                  257
Ġbranch             8478
Ġof                 286
Ġartificial         11666
Ġintelligence       4430
Ġthat               326
Ċ                   198
en                  268
ables               2977
Ġcomputers          9061
Ġto                 284
Ġunderstand         1833
,                   11
Ġinterpret          6179
,                   11
Ġand                290
Ġgenerate           7716
Ġhuman              1692
Ġlanguage           3303
.                   13
Ċ                   198
Natural             35364
Ġlanguage           3303
Ġprocessing         7587
Ġcombines           21001
Ġlingu              20280
istics              3969
,                   11
Ġcomputer           3644
Ġscience            3783
,                   11
Ġand                290
Ċ                   198
machine             30243
Ġlearning           

## Experiment 5.1b: BPE tokenizer without a pretrained model
Read text from file, split it into words, and represent each word as characters followed by an end-of-word marker. Repeatedly count adjacent symbol pairs and merge the most frequent pair (200 merges). Build the vocabulary, assign token IDs, and tokenize new words using the learned merge rules.

In [2]:
import re
from collections import Counter

with open("input_sub_word_data.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

words = re.findall(r"[a-z]+", text)
word_freq = Counter(words)

# Each word = characters + end-of-word marker
vocab = {tuple(w) + ("</w>",): freq for w, freq in word_freq.items()}

def get_pair_stats(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        for a, b in zip(word, word[1:]):
            pairs[(a, b)] += freq
    return pairs

def merge_pair(pair, vocab):
    new_vocab = {}
    for word, freq in vocab.items():
        out, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                out.append(word[i] + word[i + 1])
                i += 2
            else:
                out.append(word[i])
                i += 1
        new_vocab[tuple(out)] = freq
    return new_vocab

NUM_MERGES = 200
merges = []
for step in range(NUM_MERGES):
    pairs = get_pair_stats(vocab)
    if not pairs:
        break
    best = pairs.most_common(1)[0][0]
    vocab = merge_pair(best, vocab)
    merges.append(best)
    if step < 10:
        print(f"Merge {step + 1}: {best} -> {''.join(best)}  (freq {pairs[best]})")

# Build token -> ID mapping
symbols = set()
for word in vocab:
    symbols.update(word)
for w in word_freq:
    symbols.update(w)
symbols.add("</w>")
token2id = {"<unk>": 0}
for i, s in enumerate(sorted(symbols), start=1):
    token2id[s] = i

print("\nVocabulary size:", len(token2id))
print("Number of merges:", len(merges))

def bpe_encode(word):
    symbols = list(word.lower()) + ["</w>"]
    for pair in merges:
        i = 0
        while i < len(symbols) - 1:
            if (symbols[i], symbols[i + 1]) == pair:
                symbols[i:i + 2] = ["".join(pair)]
            else:
                i += 1
    return symbols

test_words = ["naturalization", "internationalization", "computationally", "unbelievable",
              "preprocessing", "classification", "communication", "renewable",
              "hydrokinetic", "multilingual", "understanding", "modernization"]

print("\nTokens and token IDs:")
for w in test_words:
    toks = bpe_encode(w)
    ids = [token2id.get(t, token2id["<unk>"]) for t in toks]
    print(f"{w:<22}{toks}\n{'':<22}{ids}")

Merge 1: ('e', '</w>') -> e</w>  (freq 178)
Merge 2: ('s', '</w>') -> s</w>  (freq 174)
Merge 3: ('e', 'n') -> en  (freq 125)
Merge 4: ('i', 'n') -> in  (freq 107)
Merge 5: ('d', '</w>') -> d</w>  (freq 104)
Merge 6: ('e', 'r') -> er  (freq 102)
Merge 7: ('o', 'r') -> or  (freq 97)
Merge 8: ('a', 'n') -> an  (freq 91)
Merge 9: ('t', 'h') -> th  (freq 83)
Merge 10: ('o', 'n') -> on  (freq 74)

Vocabulary size: 208
Number of merges: 200

Tokens and token IDs:
naturalization        ['nat', 'ur', 'al', 'ization</w>']
                      [129, 193, 8, 102]
internationalization  ['in', 'tern', 'ation', 'al', 'ization</w>']
                      [88, 174, 23, 8, 102]
computationally       ['com', 'pu', 't', 'ation', 'ally</w>']
                      [35, 151, 171, 23, 12]
unbelievable          ['un', 'b', 'el', 'i', 'e', 'v', 'ab', 'le</w>']
                      [189, 25, 51, 81, 47, 196, 4, 111]
preprocessing         ['pre', 'processing</w>']
                      [147, 150]
classificatio

## Experiment 5.2a: SentencePiece tokenizer using a pretrained model (T5)
Read text from file, tokenize it using the pretrained SentencePiece tokenizer from T5, and display the tokens with their token IDs. Also test the tokenizer on new words.

In [3]:
!pip install transformers sentencepiece -q

from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-small")

with open("input_sub_word_data.txt", "r", encoding="utf-8") as f:
    text = f.read()

tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("Total tokens:", len(tokens))
print("\nFirst 40 tokens and IDs:")
for t, i in list(zip(tokens, ids))[:40]:
    print(f"{t:<20}{i}")

test_words = ["naturalization", "internationalization", "computationally", "unbelievable",
              "preprocessing", "classification", "communication", "renewable",
              "hydrokinetic", "multilingual", "understanding", "modernization"]

print("\nTest words:")
for w in test_words:
    toks = tokenizer.tokenize(w)
    print(f"{w:<22}{toks}  {tokenizer.convert_tokens_to_ids(toks)}")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1319 > 512). Running this sequence through the model will result in indexing errors


Total tokens: 1319

First 40 tokens and IDs:
▁Natural            6869
▁language           1612
▁processing         3026
▁is                 19
▁                   3
a                   9
▁branch             6421
▁of                 13
▁artificial         7353
▁intelligence       6123
▁that               24
▁                   3
enables             7161
▁computers          7827
▁to                 12
▁understand         734
,                   6
▁interpret          7280
,                   6
▁and                11
▁generate           3806
▁human              936
▁language           1612
.                   5
▁Natural            6869
▁language           1612
▁processing         3026
▁                   3
combines            15256
▁                   3
linguistic          24703
s                   7
,                   6
▁computer           1218
▁science            2056
,                   6
▁and                11
▁machine            1437
▁learning           1036
▁to                 12

T

## Experiment 5.2b: SentencePiece tokenizer without a pretrained model
Train a new SentencePiece model (BPE, vocabulary size 300) on the input file, load it, and use it to tokenize a sentence and new words. Display the tokens and their token IDs.

In [4]:
!pip install sentencepiece -q

import sentencepiece as spm

# Train a new SentencePiece model on the input file
spm.SentencePieceTrainer.train(
    input="input_sub_word_data.txt",
    model_prefix="sp_lab5",
    vocab_size=300,
    model_type="bpe",          # use "unigram" for the default SentencePiece algorithm
    hard_vocab_limit=False,
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor()
sp.load("sp_lab5.model")

print("Vocabulary size:", sp.get_piece_size())

with open("input_sub_word_data.txt", "r", encoding="utf-8") as f:
    text = f.read()

first_sentence = text.split(".")[0].replace("\n", " ") + "."
print("\nSentence:", first_sentence)
print("Tokens :", sp.encode(first_sentence, out_type=str))
print("IDs    :", sp.encode(first_sentence, out_type=int))

test_words = ["naturalization", "internationalization", "computationally", "unbelievable",
              "preprocessing", "classification", "communication", "renewable",
              "hydrokinetic", "multilingual", "understanding", "modernization"]

print("\nTest words:")
for w in test_words:
    print(f"{w:<22}{sp.encode(w, out_type=str)}  {sp.encode(w, out_type=int)}")

Vocabulary size: 300

Sentence: Natural language processing is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language.
Tokens : ['▁N', 'atural', '▁language', '▁processing', '▁is', '▁a', '▁b', 'r', 'an', 'ch', '▁of', '▁a', 'r', 't', 'if', 'ic', 'i', 'al', '▁int', 'el', 'l', 'i', 'g', 'ence', '▁th', 'at', '▁', 'en', 'ab', 'l', 'es', '▁com', 'pu', 'ter', 's', '▁to', '▁understand', ',', '▁in', 'ter', 'p', 're', 't', ',', '▁and', '▁g', 'en', 'er', 'at', 'e', '▁h', 'u', 'm', 'an', '▁language', '.']
IDs    : [213, 174, 118, 178, 84, 7, 32, 259, 15, 210, 58, 7, 259, 258, 75, 46, 262, 21, 156, 61, 263, 262, 270, 153, 17, 12, 254, 4, 73, 263, 11, 112, 76, 55, 261, 22, 207, 273, 50, 55, 269, 18, 258, 273, 45, 240, 4, 6, 12, 255, 74, 265, 267, 15, 118, 275]

Test words:
naturalization        ['▁natural', 'ization']  [233, 160]
internationalization  ['▁in', 'tern', 'ational', 'ization']  [50, 155, 176, 160]
computationally       ['▁com', 'pu

## Experiment 5.3a: POS tagging using spaCy
Tag the sentence "The young student is reading an interesting book in the library." and display each unique token with its POS tag and the description of the tag.

In [5]:
!pip install spacy -q
!python -m spacy download en_core_web_sm -q

import spacy

nlp = spacy.load("en_core_web_sm")
text = "The young student is reading an interesting book in the library."
doc = nlp(text)

seen = set()
print(f"{'Token':<15}{'POS':<8}{'Tag':<8}{'Description'}")
for token in doc:
    if token.text.lower() in seen or token.is_punct:
        continue
    seen.add(token.text.lower())
    print(f"{token.text:<15}{token.pos_:<8}{token.tag_:<8}{spacy.explain(token.tag_)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Token          POS     Tag     Description
The            DET     DT      determiner
young          ADJ     JJ      adjective (English), other noun-modifier (Chinese)
student        NOUN    NN      noun, singular or mass
is             AUX     VBZ     verb, 3rd person singular present
reading        VERB    VBG     verb, gerund or present participle
an             DET     DT      determiner
interesting    ADJ     JJ      adjective (English), other noun-modifier (Chinese)
book           NOUN    NN      noun, singular or mass
in             ADP     IN      conjunction, subordina

## Experiment 5.3b: POS tagging using NLTK
Tag the same sentence using NLTK and display each unique token with its Penn Treebank POS tag and the description of the tag.

In [6]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

from nltk import word_tokenize, pos_tag

tag_desc = {
    "DT": "Determiner", "JJ": "Adjective", "NN": "Noun, singular",
    "NNS": "Noun, plural", "NNP": "Proper noun, singular", "VB": "Verb, base form",
    "VBZ": "Verb, 3rd person singular present", "VBG": "Verb, gerund/present participle",
    "VBD": "Verb, past tense", "VBN": "Verb, past participle", "VBP": "Verb, non-3rd person present",
    "IN": "Preposition/subordinating conjunction", "CC": "Coordinating conjunction",
    "PRP": "Personal pronoun", "RB": "Adverb", "TO": "to", "MD": "Modal", ".": "Punctuation"
}

text = "The young student is reading an interesting book in the library."
tagged = pos_tag(word_tokenize(text))

seen = set()
print(f"{'Token':<15}{'POS Tag':<10}{'Description'}")
for word, tag in tagged:
    if word.lower() in seen or tag == ".":
        continue
    seen.add(word.lower())
    print(f"{word:<15}{tag:<10}{tag_desc.get(tag, 'Other')}")

Token          POS Tag   Description
The            DT        Determiner
young          JJ        Adjective
student        NN        Noun, singular
is             VBZ       Verb, 3rd person singular present
reading        VBG       Verb, gerund/present participle
an             DT        Determiner
interesting    JJ        Adjective
book           NN        Noun, singular
in             IN        Preposition/subordinating conjunction
library        NN        Noun, singular


## Experiment 5.4: POS tagging with frequency using spaCy
Read text from file, run spaCy POS tagging, and display each unique token with its POS tag, tag description, and frequency. Save the result as CSV.

In [7]:
import spacy
import pandas as pd
from collections import Counter

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2_000_000

with open("input_sub_word_data.txt", "r", encoding="utf-8") as f:
    text = f.read()

doc = nlp(text)

counter = Counter()
for token in doc:
    if token.is_alpha:
        counter[(token.text.lower(), token.pos_, token.tag_)] += 1

rows = []
for (term, pos, tag), count in counter.items():
    rows.append({
        "Token": term,
        "POS": pos,
        "Tag": tag,
        "Description": spacy.explain(tag),
        "Frequency": count
    })

df = pd.DataFrame(rows).sort_values("Frequency", ascending=False).reset_index(drop=True)
df.to_csv("pos_frequency_5_4.csv", index=False)

print("Unique (token, POS) pairs:", len(df))
print(df.head(30).to_string(index=False))

Unique (token, POS) pairs: 399
       Token   POS Tag                                        Description  Frequency
         the   DET  DT                                         determiner         47
           a   DET  DT                                         determiner         28
         and CCONJ  CC                          conjunction, coordinating         27
         can   AUX  MD                              verb, modal auxiliary         23
          of   ADP  IN          conjunction, subordinating or preposition         22
          be   AUX  VB                                    verb, base form         15
          is   AUX VBZ                  verb, 3rd person singular present         14
          as   ADP  IN          conjunction, subordinating or preposition         13
         bpe PROPN NNP                              noun, proper singular         13
        word  NOUN  NN                             noun, singular or mass         12
    language  NOUN  NN            

## Experiment 5.5: Word2Vec CBOW model
Create corpus.txt, train a Word2Vec model using the CBOW architecture (sg=0), display the vector of a selected word, and find the three most similar words.

In [8]:
!pip install gensim -q

corpus_text = """Natural language processing is interesting
Natural language processing is useful
I love machine learning
Machine learning is useful
Deep learning is a part of machine learning
Artificial intelligence is changing the world
Natural language processing uses machine learning
Deep learning and artificial intelligence are related
I study natural language processing
Machine learning helps artificial intelligence"""

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(corpus_text)

from gensim.models import Word2Vec

with open("corpus.txt", "r", encoding="utf-8") as f:
    sentences = [line.lower().split() for line in f if line.strip()]

# sg=0 -> CBOW
cbow_model = Word2Vec(sentences, vector_size=50, window=2, min_count=1,
                      sg=0, epochs=200, seed=42, workers=1)

word = "learning"
print(f"Vector for '{word}':\n", cbow_model.wv[word])

print(f"\nThree most similar words to '{word}':")
for w, score in cbow_model.wv.most_similar(word, topn=3):
    print(f"{w}: {score:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 48.9 MB/s eta 0:00:00
Vector for 'learning':
 [-1.14769377e-02  1.05851330e-02  2.33525457e-03  2.97590694e-03
 -1.50652282e-04  2.05694921e-02 -2.47753393e-02  1.03978626e-02
 -1.37487827e-02 -2.18728166e-02  9.32266098e-03  2.70056985e-02
  3.96895595e-03  1.69673972e-02  8.35582707e-03  2.02155467e-02
  3.83470254e-03 -2.25487221e-02  7.96605088e-03 -5.37149003e-03
 -6.73151459e-04 -1.13366088e-02 -1.37457922e-02  1.66159775e-02
  2.38040332e-02 -3.99856763e-05 -2.45187059e-03  5.81637444e-03
 -3.74161522e-03 -1.75993750e-03 -1.07361516e-03 -4.76473244e-03
 -1.76716652e-02  6.24432880e-03  1.16029279e-02 -1.44446502e-02
  1.03355832e-02  9.16392263e-03 -1.22327805e-02  8.48266110e-03
 -1.43864111e-03  1.21485638e-02  1.60645023e-02 -3.68284783e-03
 -1.00582205e-02  1.42978970e-02 -5.08720288e-04  1.85765624e-02
  9.07086115e-03 -3.66482185e-04]

Three most similar words to 'learning':
intelligence: 0.3409
a: 0.2337
is: 0.2169


## Experiment 5.6: Word2Vec Skip-Gram model
Train a Word2Vec model on the same corpus using the Skip-Gram architecture (sg=1), retrieve the vector for "language", and display the five most similar words.

In [9]:
from gensim.models import Word2Vec

with open("corpus.txt", "r", encoding="utf-8") as f:
    sentences = [line.lower().split() for line in f if line.strip()]

# sg=1 -> Skip-Gram
sg_model = Word2Vec(sentences, vector_size=50, window=2, min_count=1,
                    sg=1, epochs=200, seed=42, workers=1)

print("Vector for 'language':\n", sg_model.wv["language"])

print("\nFive most similar words to 'language':")
for w, score in sg_model.wv.most_similar("language", topn=5):
    print(f"{w}: {score:.4f}")

Vector for 'language':
 [-2.64017610e-03  1.56689417e-02 -2.11909111e-03  1.04284706e-02
 -1.36282997e-05 -5.72224660e-03  5.75108686e-03  1.95903219e-02
 -1.06695490e-02  9.13936365e-03 -4.40732995e-03  1.36303985e-02
  8.99211317e-03  2.62717740e-03  1.00912293e-02 -4.15361347e-03
 -1.51384072e-02 -2.06049997e-02 -5.64687699e-03  1.39191579e-02
 -1.56226745e-02 -5.12496056e-03  7.07166921e-03 -1.24580106e-02
  1.59916766e-02 -1.09158596e-02  1.27718877e-02 -3.70240392e-04
 -1.12659542e-03 -1.27594899e-02 -8.06042808e-04  1.77026875e-02
 -2.05136109e-02  1.24103166e-02 -2.04425748e-03  1.09388735e-02
  3.34073696e-03 -5.51739568e-03 -9.72271804e-03  6.87587075e-03
 -8.56865104e-03  4.73325327e-03 -1.15494775e-02  7.07098469e-03
  9.78803448e-03 -1.89655181e-02  1.12826191e-02 -2.83621182e-03
  1.26009695e-02 -2.46209987e-02]

Five most similar words to 'language':
intelligence: 0.3096
uses: 0.2975
is: 0.2770
are: 0.2764
of: 0.2608


## Experiment 5.7: Cosine similarity between word pairs
Train a Word2Vec model on the same corpus and calculate the cosine similarity for the pairs (machine, learning), (natural, language), and (deep, learning). The similarity is computed manually with NumPy and checked against Gensim's built-in function.

In [10]:
import numpy as np
from gensim.models import Word2Vec

with open("corpus.txt", "r", encoding="utf-8") as f:
    sentences = [line.lower().split() for line in f if line.strip()]

model = Word2Vec(sentences, vector_size=50, window=2, min_count=1,
                 sg=1, epochs=200, seed=42, workers=1)

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

pairs = [("machine", "learning"), ("natural", "language"), ("deep", "learning")]

print(f"{'Word Pair':<25}{'Manual Cosine':<16}{'Gensim Similarity'}")
for w1, w2 in pairs:
    manual = cosine_similarity(model.wv[w1], model.wv[w2])
    built_in = model.wv.similarity(w1, w2)
    print(f"{w1 + ' - ' + w2:<25}{manual:<16.4f}{built_in:.4f}")

Word Pair                Manual Cosine   Gensim Similarity
machine - learning       0.2353          0.2353
natural - language       0.1527          0.1527
deep - learning          0.2224          0.2224
